# Частотный анализ текстов и подготовка корпуса под TF-IDF через API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DimaKorshikov/rulemma/blob/main/examples/notebooks/04_text_frequency_and_tfidf_api.ipynb)

Как быстро снять частотное ядро лемм, долю стоп-слов и подготовить чистый лемматизированный корпус для TF-IDF / текстового поиска через `POST /v1/text-stats` и `POST /v1/lemmatize` без локальной установки морфологических библиотек.

### Регистрация и доступ к HTTP API

- Личный кабинет и бесплатный API-ключ: **[rulemma.ru/account/](https://rulemma.ru/account/)** (вход по email, Яндекс ID или Google)
- Документация HTTP API: [rulemma.ru/api/](https://rulemma.ru/api/) · Тарифы и лимиты: [rulemma.ru/pricing/](https://rulemma.ru/pricing/) · OpenAPI: [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Базовый адрес API: `https://rulemma.ru/v1` · Авторизация: заголовок `Authorization: Bearer rlm_live_...` (передавать ключ в URL запрещено)

Как подключить ключ:
1. Создайте бесплатный аккаунт на [rulemma.ru/account/](https://rulemma.ru/account/) и скопируйте токен `rlm_live_...`.
2. Сохраните его в переменную окружения `RULEMMA_API_KEY` (в Google Colab удобно положить токен в панель `Secrets` под именем `RULEMMA_API_KEY` или ввести в ячейке настройки).
3. Эндпоинт `/v1/text-stats` работает по Bearer-токену (бесплатная регистрация: [rulemma.ru/account/](https://rulemma.ru/account/)); лемматизация корпуса в шаге 3 умеет запускаться и без ключа через песочницу.

### Содержание тетрадки

1. Подключение к API `rulemma.ru`
2. Частотный анализ статьи и фильтрация стоп-слов (`POST /v1/text-stats`)
3. Батчевая лемматизация корпуса документов (`POST /v1/lemmatize`)
4. Расчёт топ-терминов TF-IDF на чистом Python (плюс рецепт для `scikit-learn`)

## 1. Клиент HTTP API

In [ ]:
import json
import os
import urllib.error
import urllib.request

BASE_URL = os.environ.get("RULEMMA_API_BASE", "https://rulemma.ru/v1").rstrip("/")
API_KEY = os.environ.get("RULEMMA_API_KEY", "").strip()

# В Google Colab ключ можно автоматически прочитать из вкладки Secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore

        API_KEY = (userdata.get("RULEMMA_API_KEY") or "").strip()
    except Exception:
        pass

HAS_KEY = bool(API_KEY)
print("Базовый URL:", BASE_URL)
print(
    "Режим:",
    "боевой API с Bearer-токеном" if HAS_KEY else "публичная песочница без ключа (до 10 строк)",
)
if not HAS_KEY:
    print("Получить бесплатный API-ключ: https://rulemma.ru/account/")


def api_request(
    method: str,
    path: str,
    payload: dict | None = None,
    *,
    auth: bool = True,
    timeout: int = 20,
) -> tuple[dict, dict[str, str]]:
    """Выполняет HTTP-запрос к rulemma API без сторонних зависимостей."""
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "rulemma-jupyter-example/1.0",
    }
    if auth:
        if not API_KEY:
            raise RuntimeError(
                "Требуется RULEMMA_API_KEY. Выпустите бесплатный токен на https://rulemma.ru/account/"
            )
        headers["Authorization"] = f"Bearer {API_KEY}"

    body = None
    if payload is not None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json; charset=utf-8"

    import time as _time

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    for attempt in range(2):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                resp_headers = {k.lower(): v for k, v in resp.headers.items()}
                return json.loads(resp.read().decode("utf-8")), resp_headers
        except urllib.error.HTTPError as exc:
            raw = exc.read().decode("utf-8", errors="replace")
            if exc.code == 429 and attempt == 0:
                wait = min(12, max(1, int(exc.headers.get("Retry-After", "4"))))
                print(f"Лимит скорости (HTTP 429), пауза {wait} с...")
                _time.sleep(wait)
                continue
            raise RuntimeError(f"HTTP {exc.code} {path}: {raw}") from exc
    raise RuntimeError(f"Не удалось выполнить {method} {path}")

## 2. Частотный словарь лемм и метрики текста (`POST /v1/text-stats`)

Сервер токенизирует текст, приводит слова к леммам, исключает стоп-слова (плюс ваши `extra_stops`) и возвращает до 100 самых частых лемм с долями в процентах.

In [ ]:
article = """
Зимние кроссовки для бега по снегу отличаются глубоким протектором и мембраной.
Если вы выбираете кроссовки на зиму, обратите внимание на амортизацию подошвы,
сцепление на льду и защиту от влаги. Мембранные кроссовки сохраняют ноги сухими
при длительных пробежках, а утеплённая подкладка защищает от ветра и мороза.
"""

if HAS_KEY:
    stats, hdrs = api_request(
        "POST",
        "/text-stats",
        {"text": article, "extra_stops": ["если", "обратить", "внимание"]},
    )
    water = stats["stop_tokens"] * 100.0 / max(1, stats["total_tokens"])
    print(
        f"Всего токенов: {stats['total_tokens']} | значимых: {stats['content_tokens']} | "
        f"уникальных лемм: {stats['unique_lemmas']} | стоп-доля: {water:.1f}%"
    )
    print(f"Списано единиц: {stats['units_charged']} (остаток: {hdrs.get('x-quota-remaining', '?')})")
    print("\nТоп-10 лемм:")
    for row in stats["lemmas"][:10]:
        print(f"  {row['word']:16s} {row['count']:>3d}  ({row['share']:.1f}%)")
else:
    print("Для /v1/text-stats нужен бесплатный API-ключ: https://rulemma.ru/account/")
    print("Переходим к шагу 3 — он работает и в режиме песочницы.")

## 3. Лемматизация мини-корпуса документов для поиска и TF-IDF

Отправляем список описаний одним батчем в `POST /v1/lemmatize` и получаем нормализованные тексты и смысловые ключи.

In [ ]:
corpus = [
    "Зимние беговые кроссовки с водонепроницаемой мембраной и шипами",
    "Кофемашина автоматическая с капучинатором для зернового кофе",
    "Кофеварка рожковая для эспрессо и капучино на домашнюю кухню",
    "Аккумуляторная цепная мини-пила 18 В с шиной 15 см для сада",
    "Бензопила 2,6 л.с. с шиной 40 см для распила бревен и дров",
]

if HAS_KEY:
    lem_resp, _ = api_request("POST", "/lemmatize", {"queries": corpus, "profile": "technical"})
else:
    lem_resp, _ = api_request(
        "POST",
        "/sandbox/test",
        {"method": "lemmatize", "queries": corpus, "profile": "technical"},
        auth=False,
    )

lemmatized_docs = [item["semantic_key"] for item in lem_resp["items"]]
for idx, (orig, clean) in enumerate(zip(corpus, lemmatized_docs), 1):
    print(f"Doc {idx}: {orig}")
    print(f"   -> {clean}")

## 4. Расчёт ключевых терминов TF-IDF (без внешних зависимостей)

На лемматизированных документах TF-IDF не дробит падежи и объединяет относительные прилагательные (`зерновой` → `зерно`, `зимний` → `зима`).

In [ ]:
import math
from collections import Counter

tokenized = [doc.split() for doc in lemmatized_docs]
doc_count = len(tokenized)
df = Counter(tok for doc in tokenized for tok in set(doc))

for idx, tokens in enumerate(tokenized, 1):
    tf = Counter(tokens)
    scored = []
    for tok, cnt in tf.items():
        idf = math.log((1 + doc_count) / (1 + df[tok])) + 1.0
        scored.append((tok, (cnt / max(1, len(tokens))) * idf))
    scored.sort(key=lambda x: (-x[1], x[0]))
    top_terms = ", ".join(f"{w} ({s:.2f})" for w, s in scored[:4])
    print(f"Doc {idx} top TF-IDF: {top_terms}")

# Если установлен scikit-learn, подготовленный список сразу подаётся в TfidfVectorizer:
# from sklearn.feature_extraction.text import TfidfVectorizer
# X = TfidfVectorizer().fit_transform(lemmatized_docs)

---

## Что дальше

- Следующая тетрадка: [05_batch_csv_pipeline_and_rate_limits.ipynb](05_batch_csv_pipeline_and_rate_limits.ipynb)
- Все Jupyter-тетрадки с HTTP API: [examples/notebooks/README.md](README.md)
- Регистрация и управление API-ключами: [rulemma.ru/account/](https://rulemma.ru/account/)
- Документация эндпоинтов и форматов ошибок: [rulemma.ru/api/](https://rulemma.ru/api/) · [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Нужны миллионы строк локально без сети (Go / Python CGO / CLI)? Смотрите локальные примеры [examples/01..13](../README.md).